In [4]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

# --- Configuration ---
# Set to 90 for full paper replication, keeping it at 10 for Colab testing
TOTAL_EPOCHS = 10 
CHECKPOINT_DIR = "./model_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Initialize an unfrozen ResNet50
def get_fresh_model():
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 10)
    return model.to(device)

base_model = get_fresh_model()

# 2. Train and Save Checkpoints
def train_and_checkpoint(model, train_loader, epochs):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    print(f"Training for {epochs} epochs and saving checkpoints...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = batch[0].to(device), batch[1].to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        # Save the model state at the end of each epoch
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.pt")
        torch.save(model.state_dict(), ckpt_path)
        print(f"Saved {ckpt_path} - Loss: {running_loss/len(train_loader):.4f}")

# Execute full training run
train_and_checkpoint(base_model, train_loader, TOTAL_EPOCHS)


# 3. Compute VoG from specific checkpoints
def compute_vog_from_checkpoints(model_arch, dataloader, checkpoint_paths):
    """Calculates VoG using pre-saved model checkpoints."""
    criterion = nn.CrossEntropyLoss()
    all_grads = {i: [] for i in range(len(dataloader.dataset))}
    
    print(f"Computing VoG across {len(checkpoint_paths)} checkpoints...")
    for ckpt_path in checkpoint_paths:
        # Load the specific checkpoint
        model_arch.load_state_dict(torch.load(ckpt_path))
        model_arch.train() # Must be in train mode to get gradients
        
        for batch_idx, batch in enumerate(dataloader):
            images, labels = batch[0].to(device), batch[1].to(device)
            images.requires_grad = True
            
            model_arch.zero_grad()
            outputs = model_arch(images)
            
            # Gather pre-softmax activations for the true labels
            target_logits = outputs.gather(1, labels.view(-1, 1)).squeeze()
            target_logits.backward(torch.ones_like(target_logits))
            
            with torch.no_grad():
                pixel_grads = images.grad.abs().mean(dim=1)
                start_idx = batch_idx * dataloader.batch_size
                for i, grad in enumerate(pixel_grads):
                    all_grads[start_idx + i].append(grad.cpu().numpy())

    # Calculate Variance
    vog_scores = np.zeros(len(dataloader.dataset))
    for i in range(len(dataloader.dataset)):
        k_grads = np.stack(all_grads[i])
        pixel_variance = np.var(k_grads, axis=0) 
        vog_scores[i] = np.mean(pixel_variance) 
        
    return vog_scores

# 4. Define Checkpoint Sets (Early vs Late)
# The paper uses the first 3 epochs for Early, and last 3 for Late
early_checkpoints = [os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{i}.pt") for i in [1, 2, 3]]
late_checkpoints = [os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{i}.pt") for i in [TOTAL_EPOCHS-2, TOTAL_EPOCHS-1, TOTAL_EPOCHS]]

# Note: Dataloader shuffle MUST be False to track indices correctly
sequential_loader = DataLoader(train_subset, batch_size=32, shuffle=False)

# Compute Early and Late VoG
print("\n--- Calculating Early VoG ---")
early_vog_scores = compute_vog_from_checkpoints(get_fresh_model(), sequential_loader, early_checkpoints)

print("\n--- Calculating Late VoG ---")
late_vog_scores = compute_vog_from_checkpoints(get_fresh_model(), sequential_loader, late_checkpoints)


# 5. Evaluate the Base Model's Error Rate on VoG Subsets
def evaluate_subset_error(model, full_dataset, indices, name):
    model.eval()
    subset = Subset(full_dataset, indices)
    loader = DataLoader(subset, batch_size=32, shuffle=False)
    
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            images, labels = batch[0].to(device), batch[1].to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    error_rate = 100 - (100 * correct / total)
    print(f"{name} Error Rate: {error_rate:.2f}%")

# Sort data into High and Low percentiles (Top 25% and Bottom 25%)
total_samples = len(early_vog_scores)
top_idx = int(total_samples * 0.75)
bottom_idx = int(total_samples * 0.25)

early_sorted = np.argsort(early_vog_scores)
late_sorted = np.argsort(late_vog_scores)

print("\n=== FINAL RESULTS: THE VoG FLIP ===")
print("In Early Training, High VoG should have LOWER error (model learns them fast).")
print("In Late Training, High VoG should have HIGHER error (model struggles to memorize them).\n")

print("[Early Stage VoG Analysis]")
evaluate_subset_error(base_model, train_subset, early_sorted[:bottom_idx], "Lowest 25% Early VoG")
evaluate_subset_error(base_model, train_subset, early_sorted[top_idx:], "Highest 25% Early VoG")

print("\n[Late Stage VoG Analysis]")
evaluate_subset_error(base_model, train_subset, late_sorted[:bottom_idx], "Lowest 25% Late VoG")
evaluate_subset_error(base_model, train_subset, late_sorted[top_idx:], "Highest 25% Late VoG")

Using device: cuda


NameError: name 'train_loader' is not defined